# 训练实践
手写实现LLM只能作为深入理解大模型原理和训练细节的方法
- 手写实现 LLM 结构工作量大，难以实时跟进最新模型的结构创新；
- 从零实现的 LLM 训练无法较好地实现多卡分布式训练，训练效率较低；
- 和现有预训练 LLM 不兼容，无法使用预训练好的模型参数

要高效进行模型训练和性能优化还是得利用主流的框架。
## 模型预训练
使用HuggingFace的transformer框架，可以无需重复实现基础网络结构，通过 AutoModel 类即可一键加载任意预训练。

框架内置的 Trainer 类封装了分布式训练的核心逻辑，支持 PyTorch 原生 DDP、DeepSpeed、Megatron-LM 等多种分布式训练策略。通过简单配置训练参数，即可实现数据并行、模型并行、流水线并行的混合并行训练。

还支持与 Deepspeed、peft、wandb、Swanlab 等框架进行集成，直接通过参数设置即可无缝对接，从而快速、高效实现 LLM 训练。
### 初始化LLM
使用 transformers 的 AutoModel 类来直接初始化已经实现好的模型。以Qwen-2.5-1.5B模型为例

In [2]:
import os
# 设置环境变量，此处使用 HuggingFace 镜像网站
os.environ['HF_ENDPOINT'] = 'https://modelscope.cn'
# 下载模型
# os.system('hf download Qwen/Qwen3-0.6B --local-dir ./model/Qwen3-0.6B')
from modelscope import snapshot_download

model_dir = snapshot_download("Qwen/Qwen3-0.6B", cache_dir='./model/Qwen3-0.6B')

2026-03-08 21:06:51,642 - modelscope - INFO - Got 10 files, start to download ...


Processing 10 items:   0%|          | 0.00/10.0 [00:00<?, ?it/s]

2026-03-08 21:07:23,751 - modelscope - INFO - Download model 'Qwen/Qwen3-0.6B' successfully.
2026-03-08 21:07:23,753 - modelscope - INFO - Creating symbolic link [./model/Qwen3-0.6B/Qwen/Qwen3-0.6B].


In [ ]:
# 加载定义好的模型参数-此处以 Qwen-3-0.6B 为例
# 使用 transforemrs 的 Config 类进行加载
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# 下载参数的本地路径
model_path = "./model/Qwen3-0.6B/Qwen/Qwen3-0.6B"

config = AutoConfig.from_pretrained(model_path)
# 使用该配置生成一个定义好的模型
model = AutoModelForCausalLM.from_config(config,trust_remote_code=True)
# 加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
# 加载数据集
ds = load_dataset('json', data_files='./data/mobvoi_seq_monkey_general_open_corpus.jsonl')

# 对数据集进行 tokenize
def tokenize_function(examples):
    # 使用预先加载的 tokenizer 进行分词
    output = tokenizer([item for item in examples["text"]])
    return output


column_names = list(ds["train"].features)
# 批量处理
tokenized_datasets = ds.map(
    tokenize_function,
    batched=True,
    num_proc=10,
    remove_columns=column_names,
    load_from_cache_file=True,
    desc="Running tokenizer on dataset",
)

处理完成后的数据集会包括'input_ids', 'attention_mask'两列，分别是文本 tokenize 之后的数值序列和注意力掩码（标识是否 padding）。map 方法会通过 remove_columns 参数将原先的‘text’移除，训练中不再使用。

一次性学习多个样本的序列语义不影响模型性能，且训练数据量大、训练时间长，对训练效率要求比较高。在预训练过程中，一般会把多个文本段拼接在一起，处理成统一长度的文本块，再对每个文本块进行训练。

In [ ]:
# 预训练一般将文本拼接成固定长度的文本段
from itertools import chain

# 这里我们取块长为 2048
block_size = 2048

def group_texts(examples):
    # 将文本段拼接起来
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    # 计算拼起来的整体长度
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # 如果长度太长，进行分块
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # 按 block_size 进行切分
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    # CLM 任务，labels 和 input 是相同的
    result["labels"] = result["input_ids"].copy()
    return result

# 批量处理
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=10,
    load_from_cache_file=True,
    desc=f"Grouping texts in chunks of {block_size}",
    batch_size = 40000,
)
train_dataset = lm_datasets["train"]

### 使用Trainer训练
Trainer 封装了模型的训练逻辑，且做了较好的效率优化、可视化等工作，可以高效、便捷地完成 LLM 的训练。

In [ ]:
from transformers import TrainingArguments
from transformers import Trainer, default_data_collator
from torchdata.datapipes.iter import IterableWrapper

# 配置训练参数

training_args = TrainingArguments(
    output_dir="output",# 训练参数输出路径
    per_device_train_batch_size=4,# 训练的 batch_size
    gradient_accumulation_steps=4,# 梯度累计步数，实际 bs = 设置的 bs * 累计步数
    logging_steps=10,# 打印 loss 的步数间隔
    num_train_epochs=1,# 训练的 epoch 数
    save_steps=100, # 保存模型参数的步数间隔
    learning_rate=1e-4,# 学习率
    gradient_checkpointing=True# 开启梯度检查点
)

# 训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=IterableWrapper(train_dataset),
    eval_dataset= None,
    tokenizer=tokenizer,
    # 默认为 MLM 的 collator，使用 CLM 的 collater
    data_collator=default_data_collator
)

# 使用train()方法就可以按照配置进行训练了
# trainer.train()

完整训练流程参见pretrain_ds.py
使用以下脚本定义超参并启动Seepspeed启动训练
```bash
# 设置可见显卡
CUDA_VISIBLE_DEVICES=0,1

deepspeed pretrain.py \
    --config_name ./model/Qwen3-0.6B \
    --tokenizer_name ./model/Qwen3-0.6B \
    --train_files ./data/mobvoi_seq_monkey_general_open_corpus_small.jsonl \
    --per_device_train_batch_size 8 \
    --gradient_accumulation_steps 8 \
    --do_train \
    --output_dir ./output/pretrain \
    --evaluation_strategy  no \
    --learning_rate 1e-4 \
    --num_train_epochs 1 \
    --warmup_steps 200 \
    --logging_dir ./output/pretrain/logs \
    --logging_strategy steps \
    --logging_steps 5 \
    --save_strategy steps \
    --save_steps 100 \
    --preprocessing_num_workers 10 \
    --save_total_limit 1 \
    --seed 12 \
    --block_size 2048 \
    --bf16 \
    --gradient_checkpointing \
    --deepspeed ./ds_config_zero.json \
    --report_to swanlab
    # --resume_from_checkpoint ${output_model}/checkpoint-20400 \
```

#### Pretrain和SFT区别
核心差异在于，Pretrain 使用海量无监督文本进行训练，模型直接对文本执行“预测下一个 token”的任务；而 SFT 使用构建成对的指令对数据，模型根据输入的指令，建模后续的输出。反映到具体的训练实现上，Pretrain 会对全部 text 进行 loss 计算，要求模型对整个文本实现建模预测；而 SFT 仅对输出进行 loss 计算，不计算指令部分的 loss。

## 监督微调
相较于上一节完成的 Pretrain 代码，SFT 部分仅需要修改数据处理环节，实现对指令对数据转化为训练样本的构建，其余部分和 Pretrain 是完全一致的实现逻辑。

Qwen 系列的 Chat Template 一般有三个对话角色：System、User 和 Assistant。System 是系统提示词，负责激活模型的能力，默认为“You are a helpful assistant.”，一般不会在 SFT 过程中更改使用。User 即为用户给出的提示词，此处由于数据集中的对话角色为 “human”，我们将 “user” 修改为了“human”。Assistant 即为 LLM 给出的回复，也就是模型在 SFT 过程中需要拟合的文本。

接着，由于该数据集是一个多轮对话数据集，我们需要对多轮对话进行拼接处理，将多轮对话拼接到一个文本序列中

## 高效微调
### 高效微调方案
- Adapt Tuning。即在模型中添加 Adapter 层，在微调时冻结原参数，仅更新 Adapter 层。存在推理延迟问题，由于增加了额外参数和额外计算量，导致微调之后的模型计算速度相较原预训练模型更慢。

具体而言，其在预训练模型每层中插入用于下游任务的参数，即 Adapter 模块，在微调时冻结模型主体，仅训练特定于任务的参数。

- Prefix Tuning。该种方法固定预训练 LM，为 LM 添加可训练，任务特定的前缀，这样就可以为不同任务保存不同的前缀，微调成本也小。具体而言，在每一个输入 token 前构造一段与下游任务相关的 virtual tokens 作为 prefix，在微调时只更新 prefix 部分的参数，而其他参数冻结不变。存在固定的缺陷：模型可用序列长度减少。由于加入了 virtual tokens，占用了可用序列长度

### LoRA微调
越简单的下游任务，对应的本征秩越低。

想对于其他高效微调方法，LoRA 存在以下优势：

可以针对不同的下游任务构建小型 LoRA 模块，从而在共享预训练模型参数基础上有效地切换下游任务。
LoRA 使用自适应优化器（Adaptive Optimizer），不需要计算梯度或维护大多数参数的优化器状态，训练更有效、硬件门槛更低。
LoRA 使用简单的线性设计，在部署时将可训练矩阵与冻结权重合并，不存在推理延迟。
LoRA 与其他方法正交，可以组合。

#### 原理
LoRA 假设权重更新的过程中也有一个较低的本征秩
使用低秩分解来表示其更新
$$W_0 + \Delta W = W_0 + AB where A\in R^{d*r}, B\in R^{r*k}$$
初始化时，A使用随机高斯初始化，B使用零初始化，这样可以保证这个低秩分解的矩阵即BA是一个全零矩阵，不影响原模型的参数

Note:实际上，A和B其中一个为0即可，不过B为0的话特征提取效果更好

#### 应用于Transformer
在 Transformer 结构中，LoRA 技术主要应用在注意力模块的四个权重矩阵：$W_q$、$W_k$、$W_v$、$W_0$，而冻结 MLP 的权重矩阵。

通过消融实验发现同时调整 $W_q$ 和 $W_v$ 会产生最佳结果。

在上述条件下，可训练参数个数：$$\theta = 2 * L_{LoRA} * d_{model} *r$$
其中$L_{LoRA}$为应用LoRA权重矩阵的个数，一般r根据下游任务取4，8，16即可。
#### 代码实现
一般通过peft库实现，封装了LoRA、Adapt Tunning和P-Tuning等高效微调方法。
1. 实现流程
- 确定要使用 LoRA 的层。peft 库目前支持调用 LoRA 的层包括：nn.Linear、nn.Embedding、nn.Conv2d 三种。
- 对每一个要使用 LoRA 的层，替换为 LoRA 层。所谓 LoRA 层，实则是在该层原结果基础上增加了一个旁路，通过低秩分解（即矩阵 A 和矩阵 B）来模拟参数更新。
- 冻结原参数，进行微调，更新 LoRA 层参数。
2.确定 LoRA 层
首先需要确定 LoRA 微调参数，其中一个重要参数即是 target_modules。target_modules 一般是一个字符串列表，每一个字符串是需要进行 LoRA 的层名称。
3.替换LoRA层
对于找到的每一个目标层，会创建一个新的 LoRA 层进行替换。、
4.训练
由于在 LoRA 层中已冻结原参数，在训练中只有 A 和 B 的参数会被更新，从而实现了高效微调。训练的整体过程与原 Fine-tune 类似，此处不再赘述。由于采用了 LoRA 方式，forward 函数也会对应调整：
```python
    def forward(self, x: torch.Tensor):
        if self.disable_adapters:
            if self.r > 0 and self.merged:
                self.weight.data -= (
                    transpose(self.lora_B.weight @ self.lora_A.weight, self.fan_in_fan_out) * self.scaling
                )
                self.merged = False

            return F.linear(x, transpose(self.weight, self.fan_in_fan_out), bias=self.bias)
        '''主要分支'''
        elif self.r > 0 and not self.merged:
            result = F.linear(x, transpose(self.weight, self.fan_in_fan_out), bias=self.bias)
            if self.r > 0:
                result += self.lora_B(self.lora_A(self.lora_dropout(x))) * self.scaling
            return result
        else:
            return F.linear(x, transpose(self.weight, self.fan_in_fan_out), bias=self.bias)

```
#### 使用peft实现LoRA微调
peft进行了封装

In [ ]:
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from transformers import Trainer


# 加载基座模型
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_PATH, trust_remote_code=True
)

peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=8,
            lora_alpha=32,
            lora_dropout=0.1,
        )


model = get_peft_model(model, peft_config)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset= IterableWrapper(train_dataset),
    tokenizer=tokenizer
)
trainer.train()

## RLHF
在完成初步的指令微调后，我们还想要使模型的回答不仅正确，还能最大程度上满足人类的审美、价值观和安全标准。为此，引入了强化学习与人类反馈（Reinforcement Learning from Human Feedback, RLHF）的概念。在 RLHF 中，我们会先从人类标注者那里获得对模型回答的偏好。

数据集中需明确标识每个问题（prompt）、其对应的多个回答（completions），以及人类标注者对这些回答的选择（如标记为 "chosen" 的最佳答案与 "rejected" 的较差答案）。

